# AI-Orchestrator — Treino LoRA Qwen3.5-9B (Fases 2-3)

LoRA **bf16** (QLoRA 4-bit contraindicado pela Unsloth em Qwen3.5) sobre `unsloth/Qwen3.5-9B`,
dataset SFT de tool-calling/routing gerado na Fase 1, export GGUF Q4_K_M para Ollama local.

Runtime: **A100 40GB**. Rode as células em ordem. Pré-requisito no Drive:
`MyDrive/ai-orchestrator-dataset/orch_sft_train.jsonl` e `orch_sft_val.jsonl`.

In [ ]:
# 1) Instalação pinada — Qwen3.5 exige transformers v5 (doc oficial Unsloth)
!pip install -q --upgrade --no-cache-dir "unsloth" "unsloth_zoo"
!pip install -q --upgrade "transformers>=5.0.0,<6" "trl>=0.24,<0.30" "datasets>=3.2,<5" "accelerate>=1.2" "peft>=0.16" sentencepiece protobuf

import transformers, trl, datasets as ds_lib
import unsloth  # noqa: F401  (importa antes de transformers nos patches internos)
print("transformers:", transformers.__version__)
print("trl:", trl.__version__)
print("datasets:", ds_lib.__version__)
assert transformers.__version__.split(".")[0] == "5", "Qwen3.5 requer transformers v5 — reinicie o runtime e reinstale"


In [ ]:
# 2) Drive + dataset (formato {"messages": [...]} por linha)
from google.colab import drive
drive.mount('/content/drive')

from datasets import load_dataset

DATA_DIR = '/content/drive/MyDrive/ai-orchestrator-dataset'
raw = load_dataset('json', data_files={
    'train': f'{DATA_DIR}/orch_sft_train.jsonl',
    'val':   f'{DATA_DIR}/orch_sft_val.jsonl',
})
print(raw)
print('exemplo (roles):', [m['role'] for m in raw['train'][0]['messages']])


In [ ]:
# 3) Modelo base + LoRA bf16
# load_in_4bit=False: Unsloth contraindica QLoRA 4-bit em Qwen3.5 (LoRA 16-bit é o recomendado).
# target_modules: conjunto da doc oficial Unsloth p/ Qwen3.5 (atenção + MLP).
# As camadas lineares DeltaNet (híbridas) ficam FORA dos adapters — mesma escolha da doc/plano.
import torch
from unsloth import FastLanguageModel

MAX_SEQ = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = "unsloth/Qwen3.5-9B",
    max_seq_length  = MAX_SEQ,
    dtype           = torch.bfloat16,
    load_in_4bit    = False,
    load_in_16bit   = True,
    full_finetuning = False,
)

model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    lora_alpha     = 32,
    lora_dropout   = 0.05,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = 42,
    max_seq_length = MAX_SEQ,
)
model.print_trainable_parameters()


In [ ]:
# 4) Formatação via chat template do Qwen3.5
# Normalização necessária (formato do build_dataset.py -> formato esperado pelo template):
#  - assistant.tool_calls: dataset traz {"function": {"name", "arguments"}}; template Qwen
#    aplica `tojson` em arguments, então arguments deve ser DICT (parse se vier string)
#    e cada call ganha id/type no padrão OpenAI.
#  - role=tool: dataset usa "tool_name"; template espera "name" (+tool_call_id) e content str.
import json

def _norm_messages(messages):
    out = []
    for m in messages:
        m = dict(m)
        role = m.get("role")
        if role == "assistant" and m.get("tool_calls"):
            calls = []
            for i, tc in enumerate(m["tool_calls"]):
                fn = tc.get("function", tc)
                args = fn.get("arguments", {})
                if isinstance(args, str):
                    try:
                        args = json.loads(args)
                    except json.JSONDecodeError:
                        args = {"_raw": args}
                calls.append({
                    "id": tc.get("id") or f"call_{i}",
                    "type": "function",
                    "function": {"name": fn["name"], "arguments": args},
                })
            m["tool_calls"] = calls
            m["content"] = m.get("content") or ""
        elif role == "tool":
            m["name"] = m.pop("tool_name", m.get("name", ""))
            m.setdefault("tool_call_id", "")
            if not isinstance(m.get("content"), str):
                m["content"] = json.dumps(m.get("content"), ensure_ascii=False)
        out.append(m)
    return out

def to_text(example):
    msgs = _norm_messages(example["messages"])
    try:  # Small series: thinking off por default; garantimos explicitamente
        text = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=False, enable_thinking=False)
    except TypeError:  # template sem suporte ao kwarg
        text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    return {"text": text}

dataset = raw.map(to_text, remove_columns=raw["train"].column_names)
print(dataset)
print(dataset["train"][0]["text"][:1200])

# Sanity: tool_calls não podem ter sido duplamente escapados pelo template
assert '\\"name\\"' not in dataset["train"][0]["text"], "arguments duplamente serializados — revisar _norm_messages"


In [ ]:
# 5) SFTTrainer — 2 epochs, lr 2e-4 cosine, batch efetivo 16, eval por epoch
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = dataset["train"],
    eval_dataset  = dataset["val"],
    args = SFTConfig(
        dataset_text_field          = "text",
        max_length                  = MAX_SEQ,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 8,   # batch efetivo 16
        num_train_epochs            = 2,
        learning_rate               = 2e-4,
        lr_scheduler_type           = "cosine",
        warmup_ratio                = 0.03,
        optim                       = "adamw_8bit",
        weight_decay                = 0.01,
        bf16                        = True,
        logging_steps               = 10,
        eval_strategy               = "epoch",
        save_strategy               = "epoch",
        output_dir                  = "/content/outputs",
        seed                        = 42,
        report_to                   = "none",
    ),
)

# Mascara loss fora dos turnos do assistant (tool results viram turno "user"
# no template ChatML do Qwen, logo também ficam mascarados — comportamento desejado).
try:
    from unsloth.chat_templates import train_on_responses_only
    if "<|im_start|>" in (tokenizer.chat_template or ""):
        trainer = train_on_responses_only(
            trainer,
            instruction_part = "<|im_start|>user\n",
            response_part    = "<|im_start|>assistant\n",
        )
        print("train_on_responses_only: ATIVO")
    else:
        print("AVISO: template sem marcadores ChatML — treinando sequência completa")
except Exception as exc:
    print(f"AVISO: train_on_responses_only indisponível ({exc}) — treinando sequência completa")


In [ ]:
# 6) Treino + métricas + VRAM
import torch

torch.cuda.reset_peak_memory_stats()
stats = trainer.train()

print(f"train_loss final: {stats.metrics.get('train_loss'):.4f}")
print(f"tempo: {stats.metrics.get('train_runtime', 0)/60:.1f} min")

eval_metrics = trainer.evaluate()
print(f"val_loss final: {eval_metrics.get('eval_loss'):.4f}")

# Histórico train/val loss por step (overfit check: val subindo entre epochs = parar)
for row in trainer.state.log_history:
    if "loss" in row or "eval_loss" in row:
        print(row)

vram = torch.cuda.max_memory_reserved() / 1024**3
print(f"VRAM pico: {vram:.1f} GB / 40 GB (esperado ~22-26 GB p/ 9B bf16 LoRA)")


In [ ]:
# 7) Export: merge 16-bit -> GGUF Q4_K_M -> Drive (+Modelfile Ollama)
import glob, os, shutil

DST = '/content/drive/MyDrive/ai-orchestrator-lora'
os.makedirs(DST, exist_ok=True)

# Merge dos adapters no modelo base em 16-bit
model.save_pretrained_merged('/content/merged', tokenizer, save_method='merged_16bit')

# Conversão GGUF Q4_K_M (Unsloth compila llama.cpp na primeira chamada)
model.save_pretrained_gguf('/content/gguf', tokenizer, quantization_method='q4_k_m')

ggufs = sorted(glob.glob('/content/gguf/**/*.gguf', recursive=True),
               key=os.path.getsize, reverse=True)
assert ggufs, 'nenhum .gguf gerado — verifique logs da célula'
src = ggufs[0]
gguf_name = 'qwen3.5-9b-orch.Q4_K_M.gguf'
shutil.copy2(src, f'{DST}/{gguf_name}')
print(f'GGUF copiado: {DST}/{gguf_name} ({os.path.getsize(src)/1024**3:.1f} GB)')

# Modelfile — template ChatML do Qwen3.5 com tool-calling; thinking off por default
# (Small series não emite <think> por padrão; template não injeta bloco de reasoning).
GO_TEMPLATE = '''{{- if .Messages }}
{{- if or .System .Tools }}<|im_start|>system
{{ .System }}
{{- if .Tools }}

# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{{- range .Tools }}
{"type": "function", "function": {{ .Function }}}
{{- end }}
</tools>

For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_call>
{{- end }}<|im_end|>
{{ end }}
{{- range $i, $_ := .Messages }}
{{- $last := eq (len (slice $.Messages $i)) 1 -}}
{{- if eq .Role "user" }}<|im_start|>user
{{ .Content }}<|im_end|>
{{ else if eq .Role "assistant" }}<|im_start|>assistant
{{ if .Content }}{{ .Content }}
{{- end }}
{{- if .ToolCalls }}<tool_call>
{{ range .ToolCalls }}{"name": "{{ .Function.Name }}", "arguments": {{ .Function.Arguments }}}
{{ end }}</tool_call>
{{- end }}{{ if not $last }}<|im_end|>
{{ end }}
{{- else if eq .Role "tool" }}<|im_start|>user
<tool_response>
{{ .Content }}
</tool_response><|im_end|>
{{ end }}
{{- if and (ne .Role "assistant") $last }}<|im_start|>assistant
{{ end }}
{{- end }}
{{- else }}
{{- if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}{{ if .Prompt }}<|im_start|>user
{{ .Prompt }}<|im_end|>
{{ end }}<|im_start|>assistant
{{ end }}{{ .Response }}{{ if .Response }}<|im_end|>{{ end }}'''

TQ = '"' * 3
modelfile = (
    f'FROM ./{gguf_name}\n\n'
    f'TEMPLATE {TQ}{GO_TEMPLATE}{TQ}\n\n'
    'PARAMETER stop "<|im_start|>"\n'
    'PARAMETER stop "<|im_end|>"\n'
    'PARAMETER temperature 0.7\n'
    'PARAMETER top_p 0.8\n'
    'PARAMETER top_k 20\n'
    'PARAMETER repeat_penalty 1.0\n'
    'PARAMETER num_ctx 8192\n'
)
with open(f'{DST}/Modelfile', 'w') as fh:
    fh.write(modelfile)
print(f'Modelfile escrito em {DST}/Modelfile')


## 8) Deploy local (RTX 3060 / Ollama)

1. Baixe do Drive (`MyDrive/ai-orchestrator-lora/`) o `.gguf` (~5.5 GB) e o `Modelfile`
   para o mesmo diretório local.
2. Crie o modelo no Ollama:
   ```bash
   ollama create qwen3.5-9b-orch -f Modelfile
   ```
3. Rode os 3 gates (watchdog ativo) a partir da raiz do projeto:
   ```bash
   MODEL=qwen3.5-9b-orch python evals/eval_routing.py    # gate: >=90%
   MODEL=qwen3.5-9b-orch python evals/eval_injection.py  # gate: 0 leaks
   MODEL=qwen3.5-9b-orch python evals/eval_domains.py    # gate: >=80% por domínio
   ```
4. **Critério de adoção** (Fase 4 do plano): superar 87.5% em domains **sem regressão**
   em routing/injection. Se empatar ou piorar, manter o 9B base e documentar o
   experimento no README. Promoção (Fase 5): `.env MODEL=qwen3.5-9b-orch` + restart.